# Practical demonstration 1: the super-resolution process

*SSTA workshop, 26 November 2026. Goes with Session 2, "ADB's approach to super-resolution".*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adb-sr/sr-client/blob/main/notebooks/demo1_super_resolution.ipynb)

In this notebook you send an area and a date to ADB's super-resolution service and follow
the process from Session 2 step by step:

1. **Search**: which Sentinel-2 images exist for the area and date?
2. **The inputs**: the 8 images the model combines for one place.
3. **Stage 1**: the 8 images combined into one 2.5 m image.
4. **Stage 2**: the diffusion step adds detail.
5. **The two dials**: how much noise (Dial 1) and how many steps (Dial 2).
6. **Sharp or truthful?** Measuring what the model adds.
7. **Near-infrared**: the fourth band, and a vegetation index.
8. **Checking against real high-resolution images**: colour, near-infrared and NDVI.
9. **What comes with every result**: quality flags, sources and cost.
10. **Your own place.**

**You need** an API key (`srk_...`) and the service URL, both handed out at the workshop.
Run the cells from top to bottom with **Shift + Enter**. The super-resolution runs on the
service's GPUs, so Colab needs no GPU. The example area was run before the workshop, so its
jobs come back in about a minute and cost nothing.

## 0. Setup

In [ ]:
%pip install -q "sr-client[geo,viz] @ git+https://github.com/adb-sr/sr-client@v0.1.0"

Your API key starts with `srk_`. In Colab you can store it under **Secrets** (the key icon
on the left) as `SR_API_KEY`, and the URL as `SR_API_URL`; otherwise the cell asks for them.

In [ ]:
import getpass
import os

from sr_client import SRClient, SRError, area_around


def secret(name, prompt):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return getpass.getpass(prompt) if "KEY" in name else input(prompt)


os.environ["SR_API_KEY"] = secret("SR_API_KEY", "API key (srk_...): ")
os.environ["SR_API_URL"] = secret("SR_API_URL", "Service URL: ")

sr = SRClient()
me = sr.me()
print(f"Key {me['prefix']} ({me['owner']}): up to {me['limits']['max_area_km']} km per side, "
      f"${me['budget']['remaining_usd']:.2f} of budget left")

The service runs one model, `sen4x-v2`. It has the two stages from the slides, and the
diffusion step has the two dials: `tau` (τ, how much noise is added, Dial 1) and `nfe`
(the number of steps, Dial 2).

In [ ]:
model = sr.models()[0]
print(model["name"])
for stage, what in model["stages"].items():
    print(f"  stage {stage}: {what}")
print("dials:", {k: v for k, v in model["params"].items() if k in ("tau", "nfe")})

In [ ]:
# @title Helper functions (run this cell; you do not need to read it) { display-mode: "form" }
import threading
import time
from datetime import date

import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rioxarray
import xarray as xr
from rasterio.enums import Resampling
from rasterio.warp import transform, transform_bounds
from rasterio.windows import from_bounds
from scipy.ndimage import binary_erosion

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
EARTH_SEARCH = "https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/"


def manifest(job):
    """The job's manifest.json: scenes used per tile, parameters, cost, warnings."""
    return httpx.get(job.results()["files"]["manifest.json"], timeout=60).json()


def wait_for_stage(job, stage, poll_s=10):
    """Wait until a stage's result is published (stage 1 is published before stage 2)."""
    t0 = time.monotonic()
    while True:
        job.refresh()
        if job.status in ("failed", "cancelled"):
            raise RuntimeError(f"job {job.id} {job.status}: {job.info.get('error')}")
        try:
            if stage in job.results()["stages_done"]:
                print(f"\rstage {stage} ready after {(time.monotonic() - t0) / 60:.1f} min".ljust(60))
                return job
        except SRError as exc:
            if exc.status != 409:  # 409 = no result yet
                raise
        print(f"\r{job.status}, {(time.monotonic() - t0) / 60:.1f} min ...", end="", flush=True)
        time.sleep(poll_s)


def load(job, name):
    """A result file as an xarray DataArray in reflectance (0-1), read into memory."""
    return job.open(name).astype("float32").load()


def clip(da, lat, lon, km):
    """The km x km square around a point (works for 2.5 m and 10 m images alike)."""
    x0, y0, x1, y1 = transform_bounds("EPSG:4326", da.rio.crs, *area_around(lat, lon, km))
    return da.rio.clip_box(x0, y0, x1, y1)


def rgb(da, bands=(1, 2, 3), ceiling=0.3):
    """Natural colour (or any 3 bands) scaled for display: reflectance / ceiling."""
    a = np.moveaxis(da.sel(band=list(bands)).values, 0, -1)
    return np.clip(np.nan_to_num(a) / ceiling, 0, 1)


def show(images, titles, size=4.2, bands=(1, 2, 3), ceiling=0.3, suptitle=None):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size + 0.4),
                             squeeze=False)
    for ax, img, title in zip(axes[0], images, titles):
        ax.imshow(rgb(img, bands, ceiling) if hasattr(img, "rio") else img,
                  interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, y=1.0)
    fig.tight_layout()
    plt.show()


def to_10m(da):
    """Average 4 x 4 pixels of a 2.5 m image: what Sentinel-2 would have seen."""
    return da.coarsen(x=4, y=4, boundary="trim").mean()


def enlarge(lr_da, like):
    """A 10 m image enlarged (bicubic) to the 2.5 m grid of ``like``."""
    return lr_da.rio.write_nodata(np.nan).rio.reproject_match(like, resampling=Resampling.cubic)


def agreement(sr_da, lr_da):
    """Correlation (0-1) between a 2.5 m image averaged to 10 m and the Sentinel-2 input,
    averaged over the 4 bands. 1 = the same pattern of light and dark."""
    a = to_10m(sr_da).values
    b = lr_da.values[:, :a.shape[1], :a.shape[2]]
    ok = np.isfinite(a).all(0) & np.isfinite(b).all(0)
    return float(np.mean([np.corrcoef(a[i][ok], b[i][ok])[0, 1] for i in range(4)]))


def detail(da):
    """How much fine detail an image has: the average brightness change from one pixel to
    the next (natural-colour bands)."""
    v = np.nan_to_num(da.sel(band=[1, 2, 3]).values.mean(0))
    return float(np.mean(np.abs(np.diff(v, axis=0))) + np.mean(np.abs(np.diff(v, axis=1))))


def scene_images(scene_ids, lat, lon, km):
    """Natural colour and cloud mask of each Sentinel-2 scene around a point, read straight
    from the public archive (Element 84 Earth Search, AWS)."""
    out = []
    for sid in scene_ids:
        item = httpx.get(EARTH_SEARCH + sid, timeout=60).json()
        with rasterio.open(item["assets"]["visual"]["href"]) as r:
            win = from_bounds(*transform_bounds("EPSG:4326", r.crs, *area_around(lat, lon, km)),
                              r.transform)
            img = np.moveaxis(r.read(window=win), 0, -1)
        with rasterio.open(item["assets"]["scl"]["href"]) as r:  # scene classification, 20 m
            win = from_bounds(*transform_bounds("EPSG:4326", r.crs, *area_around(lat, lon, km)),
                              r.transform)
            scl = r.read(1, window=win, out_shape=img.shape[:2], resampling=Resampling.nearest)
        # masked like the service does: everything but vegetation, bare soil, water and
        # "unclassified" (ESA scene classes 4-7), i.e. clouds, cirrus, shadows, no data
        masked = ~np.isin(scl, (4, 5, 6, 7))
        out.append((sid, img, masked))
    return out


def ndvi(da):
    """Vegetation index: (NIR - red) / (NIR + red)."""
    red, nir = da.sel(band=1), da.sel(band=4)
    return (nir - red) / (nir + red)


# ---- section 8: checking against real high-resolution images ----------------------------
SATELLOGIC = "https://satellogic-earthview.s3.us-west-2.amazonaws.com/data/tif"


def satellogic_scene(scene):
    """Date and centre (lat, lon) of a Satellogic EarthView scene, from its name."""
    d, _, _, zone, e, n = scene.split("_")
    epsg = (32600 if zone.endswith("N") else 32700) + int(zone[:-1])
    lon, lat = transform(f"EPSG:{epsg}", "EPSG:4326", [int(e) + 192], [int(n) - 192])
    return f"{d[:4]}-{d[4:6]}-{d[6:]}", lat[0], lon[0]


def load_satellogic(scene):
    """A Satellogic EarthView scene: 384 m x 384 m at 1 m; red, green, blue, NIR;
    top-of-atmosphere reflectance (0-1). Read from the public archive on AWS."""
    d, _, _, zone, e, n = scene.split("_")
    url = (f"{SATELLOGIC}/zone={zone}/region={e}_{n}/date={d[:4]}-{d[4:6]}-{d[6:]}/"
           f"{scene}_TOA.tif")
    da = rioxarray.open_rasterio(url).astype("float32").load()
    return da.where(da > 0) / 10000


def sentinel2_image(scene_id, like):
    """One Sentinel-2 image (red, green, blue, NIR, reflectance 0-1) on the 10 m grid of
    ``like``; clouds, shadows and no data are NaN."""
    item = httpx.get(EARTH_SEARCH + scene_id, timeout=60).json()
    props = item["properties"]
    offset = 0 if props.get("earthsearch:boa_offset_applied", True) else (
        -1000 if props.get("s2:processing_baseline", "00") >= "04.00" else 0)
    bands = []
    for name in ("red", "green", "blue", "nir", "scl"):
        da = rioxarray.open_rasterio(item["assets"][name]["href"])
        x0, y0, x1, y1 = like.rio.transform_bounds(da.rio.crs)
        da = da.rio.clip_box(x0 - 50, y0 - 50, x1 + 50, y1 + 50)
        bands.append(da.rio.reproject_match(like, resampling=Resampling.nearest)[0]
                     .astype("float32"))
    clear = np.isin(bands[4], (4, 5, 6, 7)) & (bands[0] > 0)
    out = xr.concat([(b + offset) / 10000 for b in bands[:4]], dim="band")
    out = out.assign_coords(band=[1, 2, 3, 4]).where(clear)
    return out.rio.write_crs(like.rio.crs).rio.write_transform(like.rio.transform())


def _corr(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[ok], b[ok])[0, 1]) if ok.sum() > 50 else np.nan


def _moved(da, dx, dy):
    """The same image moved dx metres east and dy metres north."""
    out = da.assign_coords(x=da.x + dx, y=da.y + dy)
    return out.rio.write_transform(out.rio.transform(recalc=True))


def _averaged_onto(vhr, like):
    """The 1 m image averaged onto the grid of ``like`` (10 m or 2.5 m)."""
    return vhr.rio.write_nodata(np.nan).rio.reproject_match(like, resampling=Resampling.average)


def _align(vhr, s2):
    """Shift (metres east, north) that best lines the 1 m image up with Sentinel-2 at 10 m:
    a search in 3 m steps, then 1 m steps around the best."""
    def fit(dx, dy):
        a = _averaged_onto(_moved(vhr, dx, dy), s2).values
        return np.nanmean([_corr(a[i], s2.values[i]) for i in range(4)])

    best = max(((dx, dy) for dx in range(-24, 25, 3) for dy in range(-24, 25, 3)),
               key=lambda s: fit(*s))
    return max(((best[0] + i, best[1] + j) for i in range(-2, 3) for j in range(-2, 3)),
               key=lambda s: fit(*s))


def _match_brightness(vhr, s2):
    """Fit each band of the 1 m image to Sentinel-2 at 10 m (a straight line per band)."""
    a = _averaged_onto(vhr, s2).values
    out = vhr.copy()
    for i in range(4):
        ok = np.isfinite(a[i]) & np.isfinite(s2.values[i])
        gain, offset = np.polyfit(a[i][ok], s2.values[i][ok], 1)
        out[i] = vhr[i] * gain + offset
    return out


def _nudge(product, ref):
    """Move a product by up to one 2.5 m pixel to where it best matches the reference:
    two satellites never line up to the metre."""
    def fit(s):
        q = np.roll(product.values, s, axis=(-2, -1))
        return np.nanmean([_corr(q[b], ref.values[b]) for b in range(4)])

    s = max(((i, j) for i in (-1, 0, 1) for j in (-1, 0, 1)), key=fit)
    q = np.roll(product.values, s, axis=(-2, -1))
    if s[0]:
        q[..., 0 if s[0] > 0 else -1, :] = np.nan   # the row that wrapped around
    if s[1]:
        q[..., :, 0 if s[1] > 0 else -1] = np.nan
    return product.copy(data=q)


def fine_part(x):
    """A 2.5 m image minus its own 10 m average: the detail smaller than a Sentinel-2 pixel."""
    h, w = (x.shape[-2] // 4) * 4, (x.shape[-1] // 4) * 4
    x = x[..., :h, :w]
    m = np.nanmean(x.reshape(*x.shape[:-2], h // 4, 4, w // 4, 4), axis=(-3, -1))
    return x - np.repeat(np.repeat(m, 4, -2), 4, -1)


def compare_with_vhr(job, scene):
    """Sentinel-2, Stage 1 and Stage 2 next to a real 1 m image of the same place, all on the
    2.5 m grid of the job and cut to the 1 m image's footprint."""
    day = satellogic_scene(scene)[0]
    lr, s1, s2 = (load(job, f) for f in ("lr_reference.tif", "stage1_sr.tif", "stage2_sr.tif"))
    sid, info = min(manifest(job)["scenes"].items(), key=lambda kv: abs(
        (date.fromisoformat(kv[1]["datetime"][:10]) - date.fromisoformat(day)).days))
    s2_day = sentinel2_image(sid, lr)
    vhr = load_satellogic(scene)
    dx, dy = _align(vhr, s2_day)
    ref = _averaged_onto(_match_brightness(_moved(vhr, dx, dy), s2_day), s1)
    foot = binary_erosion(np.isfinite(ref.values).all(0), iterations=1)
    ys, xs = np.where(foot)
    # start on a 10 m pixel (4 x 4 output pixels), so fine_part's blocks are Sentinel-2's
    cut = dict(y=slice(ys.min() // 4 * 4, ys.max() + 1), x=slice(xs.min() // 4 * 4, xs.max() + 1))
    ref = ref.where(foot).isel(**cut)
    s2_date = info["datetime"][:10]
    gap = (date.fromisoformat(s2_date) - date.fromisoformat(day)).days
    images = {f"Sentinel-2 ({'same day' if gap == 0 else f'{gap:+d} days'}), enlarged":
              enlarge(s2_day, s1), "Stage 1": s1, "Stage 2": s2}
    images = {k: _nudge(v.where(foot).isel(**cut), ref) for k, v in images.items()}
    images["Satellogic, 1 m averaged to 2.5 m"] = ref
    print(f"Satellogic {day}; Sentinel-2 {s2_date}; Satellogic image moved {dx:+d} m east, "
          f"{dy:+d} m north to line up with Sentinel-2")
    return images


def show_vhr(images, title):
    fig, axes = plt.subplots(3, 4, figsize=(15, 12.5), layout="constrained")
    for j, (name, da) in enumerate(images.items()):
        axes[0, j].imshow(rgb(da), interpolation="nearest")
        axes[0, j].set_title(name, fontsize=10)
        axes[1, j].imshow(da.sel(band=4).values, cmap="gray", vmin=0, vmax=0.5,
                          interpolation="nearest")
        im = axes[2, j].imshow(ndvi(da).values, cmap="RdYlGn", vmin=-0.2, vmax=0.8,
                               interpolation="nearest")
    for ax, label in zip(axes[:, 0], ["natural colour", "near-infrared", "NDVI"]):
        ax.text(-0.04, 0.5, label, transform=ax.transAxes, rotation=90, ha="right",
                va="center", fontsize=11)
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes[2, :], orientation="horizontal", shrink=0.4, label="NDVI")
    fig.suptitle(title)
    plt.show()


def vhr_scores(images):
    """Correlation with the real image: all detail, and the detail smaller than 10 m."""
    ref = images["Satellogic, 1 m averaged to 2.5 m"]
    rows = {}
    for name, da in images.items():
        if da is ref:
            continue
        row = {}
        for label, idx in (("colour", [0, 1, 2]), ("NIR", [3])):
            row[f"{label}: all"] = np.mean([_corr(da.values[i], ref.values[i]) for i in idx])
            row[f"{label}: < 10 m"] = np.mean([_corr(fine_part(da.values[i]),
                                                     fine_part(ref.values[i])) for i in idx])
        a, b = ndvi(da).values, ndvi(ref).values
        row["NDVI: all"], row["NDVI: < 10 m"] = _corr(a, b), _corr(fine_part(a), fine_part(b))
        rows[name.split(" (")[0]] = row
    return pd.DataFrame(rows).T


class Sweep:
    """Submits several jobs one after another in the background, waiting whenever the key's
    queue is full, so the rest of the notebook can go on meanwhile.
    ``settings``: {name: (area, date, {parameters})}."""

    def __init__(self, settings):
        self.settings, self.jobs, self.errors = settings, {}, {}
        client = SRClient()  # its own connection, used only by the background thread

        def run():
            for name, (area, target_date, params) in settings.items():
                while True:
                    try:
                        self.jobs[name] = client.submit(area, target_date, **params)
                        break
                    except SRError as exc:
                        if exc.status == 429:  # queue full: try again shortly
                            time.sleep(15)
                            continue
                        self.errors[name] = exc.message
                        break

        self.thread = threading.Thread(target=run, daemon=True)
        self.thread.start()

    def status(self):
        for name in self.settings:
            job = self.jobs.get(name)
            state = (self.errors.get(name) or ("waiting to submit" if job is None
                                               else job.refresh().status))
            print(f"  {name:<22} {state}")

    def wait(self):
        while self.thread.is_alive() or any(
                j.refresh().status not in ("done", "partial", "failed", "cancelled")
                for j in self.jobs.values()):
            done = sum(j.status in ("done", "partial") for j in self.jobs.values())
            print(f"\r{done}/{len(self.settings)} done ...", end="", flush=True)
            time.sleep(10)
        print(f"\r{sum(j.status in ('done', 'partial') for j in self.jobs.values())}/"
              f"{len(self.settings)} done".ljust(40))
        for name, msg in self.errors.items():
            print(f"  {name}: not submitted: {msg}")
        return self.jobs

## 1. Search: which Sentinel-2 images are there?

`area_around(lat, lon, km)` makes a square around a point. The example is a 2 km square
over Cebu City and Mandaue. March is in the dry season, so there are few clouds.

The estimate is free. It shows how many tiles the area needs (squares of 320 m, the size the
model works on), what it would cost, and
every Sentinel-2 image the service found. It searches **45 days before and after** the date
and keeps images with **less than 40 % cloud** over the whole Sentinel-2 tile. (For the
training data we searched ±180 days, Session 2; a shorter window makes the result describe a
shorter period. You can change it with `time_window`.)

In [ ]:
LAT, LON = 10.3300, 123.9200     # Cebu City and Mandaue
DATE = "2026-03-20"              # dry season
area = area_around(LAT, LON, km=2)

est = sr.estimate(area, DATE)
print(f"{est['tiles']} tiles, about ${est['estimated_usd']:.2f}; "
      f"{len(est['scenes'])} Sentinel-2 images between {est['search_window'][0]} and "
      f"{est['search_window'][1]}")
for w in est["warnings_likely"]:
    print("warning:", w["message"])
for p in est["problems"]:
    print("cannot submit:", p)

## 2. Start the jobs

We send the job now and look at the inputs while it runs. Both stages run by default;
stage 1 is published first, after about a minute.

We also start the experiments for section 5 (the dials) in the background: seven small jobs
over a 1 km square in the middle of the area. They wait in the queue behind the main job.

In [ ]:
job = sr.submit(area, DATE)     # stage 1 + stage 2 with the default dials (tau = nfe = 250)
print("job", job.id, job.status)

SMALL = area_around(LAT, LON, km=1)
DIALS = {
    "tau 100":            dict(tau=100),
    "tau 250":            dict(tau=250),
    "tau 500":            dict(tau=500),
    "tau 999":            dict(tau=999),
    "tau 250, 10 steps":  dict(tau=250, nfe=10),
    "tau 250, 50 steps":  dict(tau=250, nfe=50),
    "tau 250, seed 1":    dict(tau=250, seed=1),
}
sweep = Sweep({name: (SMALL, DATE, p) for name, p in DIALS.items()})

## 3. The inputs, and Stage 1

As soon as stage 1 is ready, the job's manifest lists which Sentinel-2 images were used for
each tile. Below, the dots are all the images found (as in the Session 2 slide on scene
selection): the filled ones were used, the grey ones were too cloudy over this area or not
needed.

In [ ]:
wait_for_stage(job, "1")
m = manifest(job)

target = date.fromisoformat(DATE)
used = set(m["scenes"])
fig, ax = plt.subplots(figsize=(10, 3))
for s in est["scenes"]:
    d = (date.fromisoformat(s["date"][:10]) - target).days
    ax.scatter(d, s["cloud_cover"], s=60, color="#007DB7" if s["id"] in used else "none",
               edgecolors="#007DB7" if s["id"] in used else "grey")
ax.axvline(0, color="k", ls=":")
ax.set_xlabel(f"days from {DATE}")
ax.set_ylabel("cloud over the whole\nSentinel-2 tile (%)")
ax.set_title(f"{len(est['scenes'])} images found, {len(used)} used (filled)")
plt.show()

### The 8 images behind one place

For each 320 m tile the model gets 8 images, the clearest and closest in time. Here are the 8
for the tile in the middle of the area (500 m around it), read straight from the public
Sentinel-2 archive.

* Pixels outlined in red are **masked**: ESA's scene classification marks them as cloud,
  cirrus or shadow. They are filled from the other images. The classification misses some
  thin clouds and haze; look for them.
* Image 1 is the **reference**: the output follows its geometry and colours.
* If there are fewer than 8 different clear images, some are used twice (marked "repeat").

In [ ]:
# the tile nearest the middle of the area (row/col: its corner, in 10 m pixels; 32 px wide)
mid_r, mid_c = m["grid"]["height"] / 2, m["grid"]["width"] / 2
tile = min(m["tiles"], key=lambda t: (t["row"] + 16 - mid_r) ** 2 + (t["col"] + 16 - mid_c) ** 2)
scenes = scene_images(tile["scene_ids"], LAT, LON, km=0.5)

fig, axes = plt.subplots(1, len(scenes), figsize=(2.3 * len(scenes), 2.8))
for i, (ax, (sid, img, masked), d) in enumerate(zip(axes, scenes, tile["dates"]), 1):
    ax.imshow(img)
    if masked.any():
        ax.contour(masked, levels=[0.5], colors="red", linewidths=0.8)
    days = (date.fromisoformat(d) - target).days
    note = " (reference)" if i == 1 else " (repeat)" if sid in tile["scene_ids"][:i - 1] else ""
    ax.set_title(f"{i}{note}\n{days:+d} days", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Stage 1: 8 images combined into one

Stage 1 combines the 8 images into one image with 2.5 m pixels. It is sharper than one
Sentinel-2 image, but still blurry. `lr_reference.tif` is the Sentinel-2 input on the same
area: the per-pixel average of the images the model saw, at 10 m.

In [ ]:
lr = load(job, "lr_reference.tif")
s1 = load(job, "stage1_sr.tif")

show([clip(lr, LAT, LON, 0.6), clip(s1, LAT, LON, 0.6)],
     ["Sentinel-2, 10 m", "Stage 1, 2.5 m"], size=5.5)

**Stage 1 adds nothing that is not in the images.** One way to check: average each 4 × 4
block of Stage 1 back to one 10 m pixel. If Stage 1 only rearranges what the satellite
measured, this should look like the Sentinel-2 input again.

The agreement number is a correlation: 1 means the same pattern of light and dark at 10 m.
Expect a high but not perfect value: on our test areas it was 0.83 to 0.93 over cities and
0.74 to 0.77 over farmland. The Sentinel-2 input here is the average of 8 dates, and
Stage 1 makes one sharp image from them; fields that change between the dates probably
explain most of the gap over farmland.

In [ ]:
show([clip(lr, LAT, LON, 0.6), clip(to_10m(s1), LAT, LON, 0.6)],
     ["Sentinel-2, 10 m", "Stage 1, averaged back to 10 m"], size=5.5)
print(f"agreement with Sentinel-2: {agreement(s1, lr):.2f}")

## 4. Stage 2: the diffusion step adds detail

Stage 2 adds noise to the Stage 1 image and then removes it step by step. While removing the
noise, the model draws the small details it learned from 1.8 million high-resolution images
(Session 2).

In [ ]:
job.wait()
s2 = load(job, "stage2_sr.tif")
job.show()     # the whole 2 km area, both stages

In [ ]:
show([clip(lr, LAT, LON, 0.5), clip(s1, LAT, LON, 0.5), clip(s2, LAT, LON, 0.5)],
     ["Sentinel-2, 10 m", "Stage 1", "Stage 2"], size=4.6, suptitle="500 m in the middle")
print(f"agreement with Sentinel-2: Stage 1 {agreement(s1, lr):.2f}, "
      f"Stage 2 {agreement(s2, lr):.2f}")
print(f"fine detail:               Stage 1 {detail(s1):.4f}, Stage 2 {detail(s2):.4f}")

## 5. The two dials

The seven background jobs from section 2 should be done or nearly done. When they are, the
cell starts three more for section 8 in the background.

In [ ]:
sweep.status()
dial_jobs = sweep.wait()

# section 8: one job per real high-resolution image, on the day it was taken (1 km each)
CHECKS = {
    "Ningbo, China: houses, gardens and greenhouses": "20220808_022419_SN18_51N_361530_3312762",
    "near Yangon, Myanmar: factories":                "20221106_044416_SN27_47N_204846_1870691",
    "near Faisalabad, Pakistan: farmland":            "20221206_095550_SN22_43N_329899_3480761",
}
checks = Sweep({place: (area_around(lat, lon, km=1), day, {})
                for place, (day, lat, lon) in
                ((p, satellogic_scene(s)) for p, s in CHECKS.items())})

### Dial 1: how much noise (τ)

τ sets how much noise Stage 2 adds before removing it, from 0 (Stage 1 only) to 999 (almost
only noise). Little noise keeps the image close to what the satellite measured; much noise
lets the model draw more of its own detail. By default the number of steps equals τ.

In [ ]:
small = {name: load(j, "stage2_sr.tif") for name, j in dial_jobs.items()}
small["Stage 1"] = load(dial_jobs["tau 250"], "stage1_sr.tif")
lr_small = load(dial_jobs["tau 250"], "lr_reference.tif")

order = ["Stage 1", "tau 100", "tau 250", "tau 500", "tau 999"]
show([clip(small[k], LAT, LON, 0.25) for k in order],
     ["τ = 0 (Stage 1 only)", "τ = 100", "τ = 250", "τ = 500", "τ = 999"], size=3.2,
     suptitle="Dial 1: 250 m in the middle")

### Dial 2: how many steps

The details appear step by step, so the number of steps (`nfe`) also controls how sharp the
result is. Fewer steps are faster and cheaper: the cost of Stage 2 grows with the number of
steps.

In [ ]:
order = ["Stage 1", "tau 250, 10 steps", "tau 250, 50 steps", "tau 250"]
show([clip(small[k], LAT, LON, 0.25) for k in order],
     ["Stage 1", "10 steps", "50 steps", "250 steps"], size=3.6,
     suptitle="Dial 2 at τ = 250: 250 m in the middle")

print("cost of each 1 km job (USD; $0 when the result was already computed before):")
for name, j in dial_jobs.items():
    print(f"  {name:<22} {j.refresh().info['cost']['actual_usd'] or 0:.3f}")

## 6. Sharp or truthful?

Session 2 showed the trade-off: more detail, but the detail is less certain. We can measure
both sides on our own images:

* **agreement with Sentinel-2** (section 3): averaged back to 10 m, does the image still match
  what the satellite measured?
* **fine detail**: how much the brightness changes from one pixel to the next.

Look at how far each setting moves along the two axes compared with Stage 1. In our run of
this example, going from Stage 1 to τ = 999 almost doubled the fine detail (0.016 to 0.030)
while the agreement fell only from 0.88 to 0.82. Note what the agreement check *cannot* see: detail smaller than 10 m. Any detail the model draws below that
size is invisible to Sentinel-2, so it cannot be checked against it. The next experiment
shows where that detail is.

In [ ]:
rows = [("Stage 1", "Stage 1"), ("τ = 100", "tau 100"), ("τ = 250", "tau 250"),
        ("τ = 500", "tau 500"), ("τ = 999", "tau 999"),
        ("τ = 250, 10 steps", "tau 250, 10 steps"), ("τ = 250, 50 steps", "tau 250, 50 steps")]
pts = [(label, agreement(small[k], lr_small), detail(small[k])) for label, k in rows]
bicubic = enlarge(lr_small, small["Stage 1"])   # plain Sentinel-2, enlarged to 2.5 m
pts.append(("Sentinel-2, enlarged", agreement(bicubic, lr_small), detail(bicubic)))

fig, ax = plt.subplots(figsize=(8, 5))
for label, a, d in pts:
    ax.scatter(a, d, s=50, color="grey" if label.startswith("Sentinel") else "#007DB7")
    left = "steps" in label   # these sit close to τ = 100: label them on the left
    ax.annotate(label, (a, d), textcoords="offset points", xytext=(-8 if left else 8, 4),
                ha="right" if left else "left", fontsize=9)
ax.set_xlabel("agreement with Sentinel-2  (more truthful →)")
ax.set_ylabel("fine detail  (sharper →)")
plt.show()
for label, a, d in pts:
    print(f"  {label:<22} agreement {a:.3f}   detail {d:.4f}")

### Same input, different seed

The noise that Stage 2 adds is random. The `seed` fixes it: the same seed always gives the
same result. With another seed, the parts that come from the satellite stay the same, and
the parts the model drew change. The difference map shows **where the model invented
detail**.

In [ ]:
a, b = small["tau 250"], small["tau 250, seed 1"]
diff = np.abs(a.sel(band=[1, 2, 3]) - b.sel(band=[1, 2, 3])).mean("band")
show([clip(a, LAT, LON, 0.25), clip(b, LAT, LON, 0.25)], ["seed 0", "seed 1"], size=4.6)

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.imshow(clip(diff, LAT, LON, 0.25).values, cmap="magma",
          vmin=0, vmax=0.05)
ax.set_title("difference between the seeds")
ax.axis("off")
plt.show()
print(f"pixels that differ by more than 0.02 reflectance: {float((diff > 0.02).mean()):.0%}")

**What this means for users.** Large shapes (blocks, wide roads, fields) are stable; small
things (single roofs, cars, textures) can change from seed to seed. In our run of this
example, about a third of the pixels changed by more than 0.02 reflectance, mostly on roofs
and between buildings. Use super-resolved images
for questions about areas, and do not trust a single small object without checking it
(Session 3).

## 7. Near-infrared: the fourth band

The model also sharpens the near-infrared band (NIR), which our eyes cannot see. Healthy
plants reflect a lot of NIR, so it is useful for vegetation. A false-colour image shows NIR as
red, and NDVI = (NIR − red) / (NIR + red) turns it into a vegetation index.

In [ ]:
show([clip(lr, LAT, LON, 1), clip(s2, LAT, LON, 1)],
     ["Sentinel-2, false colour", "Stage 2, false colour"], bands=(4, 1, 2), ceiling=0.45,
     size=5.5)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (title, da) in zip(axes, [("Sentinel-2, 10 m", lr), ("Stage 2, 2.5 m", s2)]):
    im = ax.imshow(clip(ndvi(da), LAT, LON, 1).values,
                   cmap="RdYlGn", vmin=-0.2, vmax=0.8, interpolation="nearest")
    ax.set_title(f"NDVI, {title}")
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

## 8. Checking against real high-resolution images

So far we compared the results with Sentinel-2, which cannot show anything smaller than
10 m. To check the 2.5 m detail itself we need a real high-resolution image of the same place
on the same day.

[Satellogic EarthView](https://registry.opendata.aws/satellogic-earthview/) is a free archive
of 1 m images (red, green, blue and NIR, CC BY 4.0). Its images are the kind the model learned
from, but **these three were held out of training**: the model has never seen them or
anything within their 384 m squares. Section 5 started one 1 km job around each, for the day
the image was taken.

To compare fairly:

* the Satellogic image is averaged from 1 m to 2.5 m, the size of our pixels;
* it is lined up with the Sentinel-2 image of the same day (or the closest), at 10 m;
* its brightness is matched to that Sentinel-2 image, band by band, at 10 m. Satellogic
  measures at the top of the atmosphere, Sentinel-2 (as used here) at the ground, so their
  raw values differ. The matching uses only 10 m information, so it does not favour any of
  the images we test;
* the Sentinel-2 image we compare with is that same-day image, enlarged: what you would get
  without super-resolution.

In [ ]:
checks.status()
check_jobs = checks.wait()

place = "near Faisalabad, Pakistan: farmland"      # try the other two places as well
images = compare_with_vhr(check_jobs[place], CHECKS[place])
show_vhr(images, place)

Look at the field edges and the tracks between the fields, the roofs, and the trees. Which
of the three looks most like the real image? And which one has the real detail **in the
right place**?

The numbers below are correlations with the real image (1 = the same pattern). "all" compares
everything; "< 10 m" compares only the detail smaller than a Sentinel-2 pixel, which is the
part that super-resolution adds. Each image may move by up to one 2.5 m pixel to line up,
since two satellites never line up to the metre.

In [ ]:
shown, tables = {place: images}, {}
for place, scene in CHECKS.items():
    if place not in shown:
        shown[place] = compare_with_vhr(check_jobs[place], scene)
    tables[place.split(":")[0]] = vhr_scores(shown[place])
table = pd.concat(tables)
table.round(2)

**What we found in our run.** The pattern was the same in all three places:

* **Stage 1 has the most real detail.** Its detail smaller than 10 m matches the real image
  better than enlarged Sentinel-2 in every place, for colour, near-infrared and NDVI (NDVI:
  0.71, 0.75 and 0.83 in Ningbo, near Yangon and near Faisalabad, against 0.43, 0.57 and
  0.64 for Sentinel-2).
* **Stage 2 looks the most like the real image, but its detail is less often in the right
  place.** It scores below Stage 1, though still above Sentinel-2 in most cases. This is
  section 6 again: the diffusion step draws detail that is *plausible*, not detail that was
  *measured*.
* For maps where every pixel counts (measuring, change detection), Stage 1 is the safer
  choice; for looking and for models trained on real high-resolution images (Session 3),
  Stage 2 can work better.

**What this check cannot tell you.** Three small places, in the kind of images the model was
trained on, is a sanity check and not a full evaluation. Because the brightness is matched to
Sentinel-2, it tests patterns, not absolute values: it does not show whether an NDVI of 0.6
is really 0.6.

## 9. What comes with every result

Every job returns the images and the information you need to judge them:

| File | What it holds |
|---|---|
| `stage1_sr.tif`, `stage2_sr.tif` | the 2.5 m images: red, green, blue, NIR; reflectance × 10,000 |
| `lr_reference.tif` | the Sentinel-2 input, 10 m |
| `validity.tif` | 0 no data · 1 valid · 2 low diversity (fewer than 4 different clear images) |
| `manifest.json` | the images used for every tile, the settings, model versions, cost |

**Warnings** never stop a job, but read them: `water` (super-resolution erases detail in
shallow water and reefs; use the Sentinel-2 input there), `low_diversity`, `nodata` and
`haze`.

In [ ]:
validity = job.open("validity.tif", reflectance=False).load()
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(validity.values[0], cmap=plt.matplotlib.colors.ListedColormap(
    ["black", "#cfe8f3", "orange"]), vmin=0, vmax=2, interpolation="nearest")
ax.set_title("validity: black no data, blue valid, orange low diversity")
ax.axis("off")
plt.show()

print("warnings:", [w["code"] for w in job.warnings] or "none")
print("model:", m["model"])
print("cost:", m["cost"])
print(m["attribution"])

Download the files to open them in QGIS or ArcGIS. In Colab they appear in the file browser
on the left, under `sr_results/<job id>/`. Links last 7 days; the files are kept for 30
days.

In [ ]:
paths = job.download("sr_results")
for name, path in paths.items():
    print(f"  {path}  ({path.stat().st_size / 1e6:.1f} MB)")
print(f"budget left: ${sr.me()['budget']['remaining_usd']:.2f}")

## 10. Your own place

Change the point and the date, then run the cell. It runs stage 1 only, which takes a
minute or two and costs a few cents; add stage 2 with `mine.upgrade().wait()`.

Tips:
* keep the area at 2 km while you explore (your key allows up to 5 km);
* pick a dry-season date (in Luzon, December to May) so there are enough clear images;
* near the coast you will get the `water` warning: the sea and reefs are not reliable.

In [ ]:
MY_LAT, MY_LON, MY_DATE = 14.6760, 121.0437, "2026-04-15"   # Quezon City; change me

mine = sr.submit(area_around(MY_LAT, MY_LON, km=2), MY_DATE, stages=["1"])
mine.wait()
mine.show()

In [ ]:
mine.upgrade().wait()
mine.show()

---
Results contain modified Copernicus Sentinel data. Sentinel-2 scenes are read from the public
archive on AWS (Element 84 Earth Search).